# Physics-Informed Neural Networks pour la modélisation pharmacocinétique (PBPK)

Ce notebook accompagne le rapport *"Physics-Informed Neural Networks pour la modélisation pharmacocinétique : application aux modèles PBPK de distribution de médicaments"* (Polytech Lyon, MAM5).

Les **Physics-Informed Neural Networks (PINNs)** combinent apprentissage profond et contraintes physiques (équations différentielles) pour résoudre simultanément :
- le **problème direct** : prédire l'évolution des concentrations dans le temps,
- le **problème inverse** : estimer les paramètres physiologiques du modèle à partir d'observations partielles et bruitées.

On compare deux formulations d'un même système à 4 compartiments corporels (sang cérébral `Cbb`, tissu cérébral `Cbm`, LCR crânien `Cccsf`, LCR spinal `Cscsf`) :
1. un **modèle linéaire à 9 paramètres** (approche phénoménologique),
2. un **modèle PBPK biophysique à 4 paramètres** (approche mécanistique, basée sur des volumes physiologiques réels).

**Plan du notebook :**
1. Modèle linéaire (9 paramètres)
2. Modèle PBPK : vérification directe (sans PINN)
3. PINN sur le modèle PBPK : première estimation (3 paramètres)
4. Analyse d'identifiabilité : `Qbb` fixé vs estimé
5. Extension : ajout de quelques observations sur `Cbm`
6. Modèle PBPK final : estimation de 4 paramètres (avec amplitude `A`)


## 1. Modèle linéaire (9 paramètres)

Le système d'EDO linéaire décrit les échanges entre les 4 compartiments avec 9 taux de transfert `k_ij` à estimer. On génère des données synthétiques bruitées (bruit gaussien, σ = 0.02), on n'observe que 2 des 4 compartiments (`Cbb` et `Ccc`), et on entraîne un PINN à reconstruire l'ensemble des trajectoires tout en estimant les 9 paramètres. Les paramètres sont reparamétrés avec une fonction sigmoïde bornée pour rester dans une plage physiologiquement plausible (`[0.3×vrai, 3×vrai]`).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# 1) Données synthétiques
# =========================
t0, t1 = 0.0, 48.0
N = 200
t_np = np.linspace(t0, t1, N).astype(np.float32)

def Cart_np(t):
    A, tau = 1.0, 8.0
    return A * np.exp(-t / tau)

Cart_vals_np = Cart_np(t_np).astype(np.float32)

params_true = dict(
    k_in=0.8,
    k12=0.6, k21=0.25,
    k13=0.35, k31=0.15,
    k24=0.20, k42=0.10,
    k34=0.18, k43=0.12,
)

def rhs_true(ti, y, p):
    Cbb, Cbm, Ccc, Csc = y
    Cart_t = Cart_np(ti)
    k_in = p["k_in"]
    k12, k21 = p["k12"], p["k21"]
    k13, k31 = p["k13"], p["k31"]
    k24, k42 = p["k24"], p["k42"]
    k34, k43 = p["k34"], p["k43"]

    dCbb = k_in*Cart_t - k12*Cbb + k21*Cbm - k13*Cbb + k31*Ccc
    dCbm = k12*Cbb - k21*Cbm - k24*Cbm + k42*Csc
    dCcc = k13*Cbb - k31*Ccc - k34*Ccc + k43*Csc
    dCsc = k24*Cbm + k34*Ccc - (k42 + k43)*Csc
    return np.array([dCbb, dCbm, dCcc, dCsc], dtype=np.float32)

Y = np.zeros((N, 4), dtype=np.float32)
Y[0] = np.array([0, 0, 0, 0], dtype=np.float32)
for i in range(N - 1):
    dt = t_np[i+1] - t_np[i]
    Y[i+1] = Y[i] + dt * rhs_true(float(t_np[i]), Y[i], params_true)

sigma = 0.02
rng = np.random.default_rng(0)
C_obs_np = (Y + sigma * rng.standard_normal(Y.shape)).astype(np.float32)

# =========================
# 2) Observations partielles
# =========================
OBS_MASK = np.array([1, 0, 1, 0], dtype=np.float32)  # Cbb, Ccc
OBS_MASK_t = torch.tensor(OBS_MASK.reshape(1, 4), device=device)

# =========================
# 3) Tenseurs + Cart interp
# =========================
t_data = torch.tensor(t_np.reshape(-1, 1), device=device)
C_data = torch.tensor(C_obs_np, device=device)
Cart_data = torch.tensor(Cart_vals_np.reshape(-1, 1), device=device)

t_grid = t_data.detach().clone()
Cart_grid = Cart_data.detach().clone()

def Cart_torch(t_query):
    tq = t_query.squeeze(1)
    tg = t_grid.squeeze(1)
    idx = torch.searchsorted(tg, tq, right=False).clamp(1, len(tg)-1)
    t0i = tg[idx-1]; t1i = tg[idx]
    y0i = Cart_grid.squeeze(1)[idx-1]; y1i = Cart_grid.squeeze(1)[idx]
    w = (tq - t0i) / (t1i - t0i + 1e-12)
    y = y0i + w * (y1i - y0i)
    return y.unsqueeze(1)

# =========================
# 4) Réseau
# =========================
class MLP(nn.Module):
    def __init__(self, in_dim=1, out_dim=4, width=64, depth=5):
        super().__init__()
        layers = [nn.Linear(in_dim, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        return self.net(t)

model = MLP().to(device)

# =========================
# 5) Paramètres entraînables BORNÉS (sigmoid)
# =========================
def bounded(raw, lo, hi):
    return lo + (hi - lo) * torch.sigmoid(raw)

# Bornes "larges" (autour des vrais) : [0.3*true, 3*true]
# (suffisant pour démo; en PBPK ce sera physiologique)
bounds = {}
for k, v in params_true.items():
    lo = 0.3 * v
    hi = 3.0 * v
    bounds[k] = (lo, hi)

raw_params = nn.ParameterDict({k: nn.Parameter(torch.tensor(0.0, device=device)) for k in params_true.keys()})

def get_params():
    return {k: bounded(raw_params[k], *bounds[k]) for k in raw_params.keys()}

# =========================
# 6) ODE RHS
# =========================
def ode_rhs_torch(t, C):
    p = get_params()
    Cbb = C[:, 0:1]
    Cbm = C[:, 1:2]
    Ccc = C[:, 2:3]
    Csc = C[:, 3:4]
    Cart = Cart_torch(t)

    dCbb = p["k_in"]*Cart - p["k12"]*Cbb + p["k21"]*Cbm - p["k13"]*Cbb + p["k31"]*Ccc
    dCbm = p["k12"]*Cbb - p["k21"]*Cbm - p["k24"]*Cbm + p["k42"]*Csc
    dCcc = p["k13"]*Cbb - p["k31"]*Ccc - p["k34"]*Ccc + p["k43"]*Csc
    dCsc = p["k24"]*Cbm + p["k34"]*Ccc - (p["k42"] + p["k43"])*Csc
    return torch.cat([dCbb, dCbm, dCcc, dCsc], dim=1)

def dCdt_autodiff(t, C_pred):
    grads = []
    for j in range(C_pred.shape[1]):
        g = torch.autograd.grad(C_pred[:, j].sum(), t, create_graph=True, retain_graph=True)[0]
        grads.append(g)
    return torch.cat(grads, dim=1)

# =========================
# 7) Collocation
# =========================
def sample_collocation(n=2000, n_on_data=500, n_near0=200):
    t_uni = np.random.uniform(t0, t1, size=(n, 1)).astype(np.float32)
    idx = np.random.choice(len(t_np), size=n_on_data, replace=True)
    t_on = t_np[idx].reshape(-1, 1).astype(np.float32)
    t_0 = np.random.uniform(t0, t0 + 2.0, size=(n_near0, 1)).astype(np.float32)
    tt = np.vstack([t_uni, t_on, t_0])
    return torch.tensor(tt, device=device)

# =========================
# 8) Losses
# =========================
def loss_data():
    C_pred = model(t_data)
    diff = (C_pred - C_data) * OBS_MASK_t
    return (diff**2).mean()

def loss_ic():
    t_init = torch.tensor([[t0]], device=device, requires_grad=True)
    C0_pred = model(t_init)
    return (C0_pred**2).mean()

def loss_ode(n_colloc=2000):
    t_col = sample_collocation(n_colloc).requires_grad_(True)
    C_pred = model(t_col)
    dCdt = dCdt_autodiff(t_col, C_pred)
    rhs = ode_rhs_torch(t_col, C_pred)
    return ((dCdt - rhs)**2).mean()

# =========================
# 9) Entraînement
# =========================
opt = torch.optim.Adam(list(model.parameters()) + list(raw_params.parameters()), lr=1e-3)
w_data, w_ic, w_ode = 10.0, 1.0, 2.0

for it in range(1, 20001):
    opt.zero_grad()
    ld = loss_data()
    li = loss_ic()
    lo = loss_ode(2000)
    loss = w_data*ld + w_ic*li + w_ode*lo
    loss.backward()
    opt.step()

    if it % 1000 == 0:
        with torch.no_grad():
            est = {k: float(get_params()[k].cpu()) for k in params_true.keys()}
        print(f"it={it:5d} | loss={loss.item():.4e} | Ld={ld.item():.2e} Li={li.item():.2e} Lo={lo.item():.2e}")
        print("   params:", {k: round(est[k], 4) for k in est})

# =========================
# 10) Plots + tableau paramètres
# =========================
with torch.no_grad():
    C_pred = model(t_data).cpu().numpy()

labels = ["Cbb", "Cbm", "Ccc", "Csc"]

plt.figure()
for j in range(4):
    plt.plot(t_np, Y[:, j], label=f"{labels[j]} true")
plt.legend(); plt.title("Trajectoires vraies"); plt.xlabel("t"); plt.show()

plt.figure()
for j in range(4):
    plt.plot(t_np, C_obs_np[:, j], label=f"{labels[j]} obs", alpha=0.6)
plt.legend(); plt.title("Observations (bruitées)"); plt.xlabel("t"); plt.show()

plt.figure()
for j in range(4):
    plt.plot(t_np, C_pred[:, j], label=f"{labels[j]} pred")
plt.legend(); plt.title("Prédiction PINN (paramètres bornés, observations partielles)"); plt.xlabel("t"); plt.show()

est = {k: float(get_params()[k].cpu()) for k in params_true.keys()}

print("\nParamètres vrais vs estimés (bornés)")
for k in params_true.keys():
    print(f"{k:>4s} | true={params_true[k]:.4f} | est={est[k]:.4f} | err={abs(est[k]-params_true[k]):.4f}")


## 2. Modèle PBPK : vérification directe (sans PINN)

Avant d'entraîner un PINN, on vérifie que le modèle PBPK biophysique (flux entre compartiments pondérés par des perméabilités et des débits, normalisés par les volumes anatomiques) produit une dynamique stable et physiologiquement cohérente, en intégrant simplement les équations avec un schéma d'Euler explicite.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Temps
# =========================
t0, t1 = 0.0, 48.0
N = 2000
t = np.linspace(t0, t1, N)
dt = t[1] - t[0]

# =========================
# Entrée plasma
# =========================
def Cart(t):
    A, tau = 1.0, 8.0
    return A * np.exp(-t / tau)

# =========================
# Paramètres PBPK (réalistes)
# =========================
Vbb   = 0.105   # L
Vbm   = 1.25
Vccsf = 0.175
Vscsf = 0.055

Qbb   = 0.5     # débit perfusion
Qcsf  = 0.05    # débit CSF

PSbb  = 0.15    # perméabilité BBB
PScsf = 0.08    # perméabilité CSF

# =========================
# État initial
# =========================
C = np.zeros((N, 4))  # [Cbb, Cbm, Cccsf, Cscsf]

# =========================
# PBPK forward (Euler stable)
# =========================
for i in range(N - 1):
    Cbb, Cbm, Ccc, Csc = C[i]

    # Flux
    F_plasma = Qbb * (Cart(t[i]) - Cbb)
    F_bb_bm  = PSbb * (Cbb - Cbm)
    F_bm_cc  = PScsf * (Cbm - Ccc)
    F_cc_sc  = Qcsf * (Ccc - Csc)
    F_sc_out = Qcsf * Csc

    dCbb = (F_plasma - F_bb_bm) / Vbb
    dCbm = (F_bb_bm - F_bm_cc) / Vbm
    dCcc = (F_bm_cc - F_cc_sc) / Vccsf
    dCsc = (F_cc_sc - F_sc_out) / Vscsf

    C[i+1] = C[i] + dt * np.array([dCbb, dCbm, dCcc, dCsc])

# =========================
# Plots
# =========================
labels = ["Cbb", "Cbm", "Cccsf", "Cscsf"]
plt.figure()
for j in range(4):
    plt.plot(t, C[:, j], label=labels[j])
plt.legend()
plt.title("PBPK forward stable (sans PINN)")
plt.xlabel("t")
plt.show()


## 3. PINN sur le modèle PBPK : première estimation (3 paramètres)

On entraîne un PINN à estimer 3 paramètres physiologiques du modèle PBPK (`PSbb`, `PScsf`, `Qbb`), à partir d'observations partielles (`Cbb` et `Cccsf`). L'entraînement se fait en deux phases : Adam pour une convergence rapide, puis L-BFGS pour affiner la solution.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# =========================
# 0) Reproductibilité + device
# =========================
torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# 1) Temps
# =========================
t0, t1 = 0.0, 48.0
N = 200
t_np = np.linspace(t0, t1, N).astype(np.float32)
t_data = torch.tensor(t_np.reshape(-1, 1), device=device)

# =========================
# 2) Entrée plasma Cart(t)
# =========================
def Cart_np(t):
    return np.exp(-t / 8.0)

Cart_vals_np = Cart_np(t_np).astype(np.float32)
Cart_data = torch.tensor(Cart_vals_np.reshape(-1, 1), device=device)

def Cart_torch(t_query):
    """
    Interpolation linéaire torch sur la grille (t_data, Cart_data).
    t_query: (M,1)
    returns: (M,1)
    """
    tq = t_query.squeeze(1)
    tg = t_data.squeeze(1)
    idx = torch.searchsorted(tg, tq, right=False).clamp(1, len(tg) - 1)
    t0i = tg[idx - 1]; t1i = tg[idx]
    y0  = Cart_data.squeeze(1)[idx - 1]
    y1  = Cart_data.squeeze(1)[idx]
    w = (tq - t0i) / (t1i - t0i + 1e-12)
    y = y0 + w * (y1 - y0)
    return y.unsqueeze(1)

# =========================
# 3) Paramètres FIXES (volumes + Qcsf)
# =========================
Vbb, Vbm, Vccsf, Vscsf = 0.105, 1.25, 0.175, 0.055
Qcsf = 0.05

# =========================
# 4) Paramètres à ESTIMER (bornés)
# =========================
def bounded(raw, lo, hi):
    return lo + (hi - lo) * torch.sigmoid(raw)

raw_PSbb  = nn.Parameter(torch.tensor(0.0, device=device))
raw_PScsf = nn.Parameter(torch.tensor(0.0, device=device))
raw_Qbb   = nn.Parameter(torch.tensor(0.0, device=device))

def get_params():
    return {
        "PSbb":  bounded(raw_PSbb,  0.05, 0.4),
        "PScsf": bounded(raw_PScsf, 0.02, 0.2),
        "Qbb":   bounded(raw_Qbb,   0.1,  1.0),
    }

# Vrais paramètres utilisés pour générer les données synthétiques
true_params = {"PSbb": 0.15, "PScsf": 0.08, "Qbb": 0.5}

def print_params(tag):
    p = get_params()
    out = {k: float(p[k].detach().cpu()) for k in p}
    errs = {k: out[k] - true_params[k] for k in out}
    print(tag)
    print("  params:", out)
    print("  errors:", errs)

# =========================
# 5) Réseau PINN : t -> (Cbb, Cbm, Cccsf, Cscsf)
# =========================
class MLP(nn.Module):
    def __init__(self, width=64, depth=4):
        super().__init__()
        layers = [nn.Linear(1, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 4)]
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        return self.net(t)

model = MLP().to(device)

def dCdt_autodiff(t, C_pred):
    grads = []
    for j in range(4):
        g = torch.autograd.grad(C_pred[:, j].sum(), t, create_graph=True, retain_graph=True)[0]
        grads.append(g)
    return torch.cat(grads, dim=1)

# =========================
# 6) RHS PBPK stable (flux)
# =========================
def pbpk_rhs_torch(t, C):
    p = get_params()
    Cbb = C[:, 0:1]
    Cbm = C[:, 1:2]
    Ccc = C[:, 2:3]
    Csc = C[:, 3:4]
    Cart = Cart_torch(t)

    F_plasma = p["Qbb"] * (Cart - Cbb)
    F_bb_bm  = p["PSbb"] * (Cbb - Cbm)
    F_bm_cc  = p["PScsf"] * (Cbm - Ccc)
    F_cc_sc  = Qcsf * (Ccc - Csc)
    F_sc_out = Qcsf * Csc

    dCbb = (F_plasma - F_bb_bm) / Vbb
    dCbm = (F_bb_bm - F_bm_cc) / Vbm
    dCcc = (F_bm_cc - F_cc_sc) / Vccsf
    dCsc = (F_cc_sc - F_sc_out) / Vscsf

    return torch.cat([dCbb, dCbm, dCcc, dCsc], dim=1)

# =========================
# 7) Générer données synthétiques (forward Euler stable)
# =========================
# On génère avec les vrais paramètres (true_params)
C_true = np.zeros((N, 4), dtype=np.float32)

for i in range(N - 1):
    dt = t_np[i+1] - t_np[i]
    Cbb, Cbm, Ccc, Csc = C_true[i]
    Cart_i = Cart_np(t_np[i])

    F_plasma = true_params["Qbb"] * (Cart_i - Cbb)
    F_bb_bm  = true_params["PSbb"] * (Cbb - Cbm)
    F_bm_cc  = true_params["PScsf"] * (Cbm - Ccc)
    F_cc_sc  = Qcsf * (Ccc - Csc)
    F_sc_out = Qcsf * Csc

    dCbb = (F_plasma - F_bb_bm) / Vbb
    dCbm = (F_bb_bm - F_bm_cc) / Vbm
    dCcc = (F_bm_cc - F_cc_sc) / Vccsf
    dCsc = (F_cc_sc - F_sc_out) / Vscsf

    C_true[i+1] = C_true[i] + dt * np.array([dCbb, dCbm, dCcc, dCsc], dtype=np.float32)

# Observations bruitées (option)
sigma = 0.00  # mets 0.02 si tu veux du bruit
rng = np.random.default_rng(0)
C_obs_np = (C_true + sigma * rng.standard_normal(C_true.shape)).astype(np.float32)
C_obs = torch.tensor(C_obs_np, device=device)

# Observations partielles : Cbb et Cccsf
OBS_MASK = torch.tensor([[1, 0, 1, 0]], device=device)

# =========================
# 8) Losses
# =========================
def loss_data():
    C_pred = model(t_data)
    diff = (C_pred - C_obs) * OBS_MASK
    return (diff**2).mean()

def loss_ic():
    t_init = torch.tensor([[t0]], device=device, requires_grad=True)
    C0_pred = model(t_init)
    return (C0_pred**2).mean()

def sample_collocation(n=2000):
    tt = np.random.uniform(t0, t1, size=(n, 1)).astype(np.float32)
    return torch.tensor(tt, device=device)

def loss_ode(n_col=2000):
    t_col = sample_collocation(n_col).requires_grad_(True)
    C_pred = model(t_col)
    res = dCdt_autodiff(t_col, C_pred) - pbpk_rhs_torch(t_col, C_pred)
    return (res**2).mean()

# =========================
# 9) Phase 1: Adam
# =========================
opt = torch.optim.Adam(list(model.parameters()) + [raw_PSbb, raw_PScsf, raw_Qbb], lr=1e-3)
w_data, w_ic, w_ode = 10.0, 1.0, 2.0

for it in range(1, 15001):
    opt.zero_grad()
    Ld = loss_data()
    Li = loss_ic()
    Lo = loss_ode(2000)
    loss = w_data*Ld + w_ic*Li + w_ode*Lo
    loss.backward()
    opt.step()

    if it % 2000 == 0:
        print(f"it={it:5d} | loss={loss.item():.4e} | Ld={Ld.item():.2e} Li={Li.item():.2e} Lo={Lo.item():.2e}")
        print_params("  Adam snapshot")

print_params("After Adam")

# =========================
# 10) Phase 2: LBFGS
# =========================
lbfgs = torch.optim.LBFGS(
    list(model.parameters()) + [raw_PSbb, raw_PScsf, raw_Qbb],
    lr=1.0,
    max_iter=500,
    history_size=50,
    line_search_fn="strong_wolfe"
)

def closure():
    lbfgs.zero_grad()
    Ld = loss_data()
    Li = loss_ic()
    Lo = loss_ode(4000)  # un peu plus de collocation aide LBFGS
    loss = w_data*Ld + w_ic*Li + w_ode*Lo
    loss.backward()
    return loss

loss_lbfgs = lbfgs.step(closure)
print("LBFGS loss returned:", float(loss_lbfgs))
print_params("After LBFGS")

# =========================
# 11) Plots finaux
# =========================
with torch.no_grad():
    C_pred_np = model(t_data).cpu().numpy()

labels = ["Cbb", "Cbm", "Cccsf", "Cscsf"]

plt.figure()
for j in range(4):
    plt.plot(t_np, C_true[:, j], label=f"{labels[j]} true")
plt.legend(); plt.title("PBPK true (forward)"); plt.xlabel("t"); plt.show()

plt.figure()
for j in range(4):
    plt.plot(t_np, C_obs_np[:, j], label=f"{labels[j]} obs", alpha=0.7)
plt.legend(); plt.title("PBPK observations"); plt.xlabel("t"); plt.show()

plt.figure()
for j in range(4):
    plt.plot(t_np, C_pred_np[:, j], label=f"{labels[j]} pred")
plt.legend(); plt.title("iPINN prediction"); plt.xlabel("t"); plt.show()


## 4. Analyse d'identifiabilité : `Qbb` fixé vs estimé

Un problème classique des modèles PBPK est la **non-identifiabilité** : plusieurs paramètres peuvent être corrélés, rendant leur estimation individuelle instable même si le modèle reproduit bien les données. On teste ici l'effet de fixer `Qbb` à sa valeur vraie (au lieu de l'estimer) sur la qualité de l'estimation de `PSbb` et `PScsf`. Une différence significative entre les deux estimations confirmerait un couplage entre `Qbb` et les autres paramètres, ce qui est exactement ce que révèle l'analyse de corrélation du rapport (ρ(πbb, Qbb) = 0.87).

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# Temps
# =========================
t0, t1 = 0.0, 48.0
N = 200
t_np = np.linspace(t0, t1, N).astype(np.float32)
t_data = torch.tensor(t_np.reshape(-1, 1), device=device)

# =========================
# Entrée plasma Cart(t)
# =========================
def Cart_np(t):
    return np.exp(-t / 8.0)

Cart_vals_np = Cart_np(t_np).astype(np.float32)
Cart_data = torch.tensor(Cart_vals_np.reshape(-1, 1), device=device)

def Cart_torch(t_query):
    tq = t_query.squeeze(1)
    tg = t_data.squeeze(1)
    idx = torch.searchsorted(tg, tq, right=False).clamp(1, len(tg) - 1)
    t0i = tg[idx - 1]; t1i = tg[idx]
    y0  = Cart_data.squeeze(1)[idx - 1]
    y1  = Cart_data.squeeze(1)[idx]
    w = (tq - t0i) / (t1i - t0i + 1e-12)
    y = y0 + w * (y1 - y0)
    return y.unsqueeze(1)

# =========================
# Paramètres FIXES
# =========================
Vbb, Vbm, Vccsf, Vscsf = 0.105, 1.25, 0.175, 0.055
Qcsf = 0.05

# Qbb fixé (test identifiabilité)
Qbb_fixed = 0.5

# =========================
# Paramètres à ESTIMER (bornés)
# =========================
def bounded(raw, lo, hi):
    return lo + (hi - lo) * torch.sigmoid(raw)

raw_PSbb  = nn.Parameter(torch.tensor(0.0, device=device))
raw_PScsf = nn.Parameter(torch.tensor(0.0, device=device))

def get_params():
    return {
        "PSbb":  bounded(raw_PSbb,  0.05, 0.4),
        "PScsf": bounded(raw_PScsf, 0.02, 0.2),
        "Qbb":   torch.tensor(Qbb_fixed, device=device),
    }

true_params = {"PSbb": 0.15, "PScsf": 0.08, "Qbb": 0.5}

def print_params(tag):
    p = get_params()
    out = {k: float(p[k].detach().cpu()) for k in p}
    errs = {k: out[k] - true_params[k] for k in out}
    print(tag)
    print("  params:", out)
    print("  errors:", errs)

# =========================
# Réseau PINN
# =========================
class MLP(nn.Module):
    def __init__(self, width=64, depth=4):
        super().__init__()
        layers = [nn.Linear(1, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 4)]
        self.net = nn.Sequential(*layers)
    def forward(self, t):
        return self.net(t)

model = MLP().to(device)

def dCdt_autodiff(t, C_pred):
    grads = []
    for j in range(4):
        g = torch.autograd.grad(C_pred[:, j].sum(), t, create_graph=True, retain_graph=True)[0]
        grads.append(g)
    return torch.cat(grads, dim=1)

# =========================
# RHS PBPK
# =========================
def pbpk_rhs_torch(t, C):
    p = get_params()
    Cbb = C[:, 0:1]
    Cbm = C[:, 1:2]
    Ccc = C[:, 2:3]
    Csc = C[:, 3:4]
    Cart = Cart_torch(t)

    F_plasma = p["Qbb"] * (Cart - Cbb)
    F_bb_bm  = p["PSbb"] * (Cbb - Cbm)
    F_bm_cc  = p["PScsf"] * (Cbm - Ccc)
    F_cc_sc  = Qcsf * (Ccc - Csc)
    F_sc_out = Qcsf * Csc

    dCbb = (F_plasma - F_bb_bm) / Vbb
    dCbm = (F_bb_bm - F_bm_cc) / Vbm
    dCcc = (F_bm_cc - F_cc_sc) / Vccsf
    dCsc = (F_cc_sc - F_sc_out) / Vscsf

    return torch.cat([dCbb, dCbm, dCcc, dCsc], dim=1)

# =========================
# Données synthétiques (forward)
# =========================
C_true = np.zeros((N, 4), dtype=np.float32)
for i in range(N - 1):
    dt = t_np[i+1] - t_np[i]
    Cbb, Cbm, Ccc, Csc = C_true[i]
    Cart_i = Cart_np(t_np[i])

    F_plasma = true_params["Qbb"] * (Cart_i - Cbb)
    F_bb_bm  = true_params["PSbb"] * (Cbb - Cbm)
    F_bm_cc  = true_params["PScsf"] * (Cbm - Ccc)
    F_cc_sc  = Qcsf * (Ccc - Csc)
    F_sc_out = Qcsf * Csc

    dCbb = (F_plasma - F_bb_bm) / Vbb
    dCbm = (F_bb_bm - F_bm_cc) / Vbm
    dCcc = (F_bm_cc - F_cc_sc) / Vccsf
    dCsc = (F_cc_sc - F_sc_out) / Vscsf

    C_true[i+1] = C_true[i] + dt * np.array([dCbb, dCbm, dCcc, dCsc], dtype=np.float32)

sigma = 0.00
rng = np.random.default_rng(0)
C_obs_np = (C_true + sigma * rng.standard_normal(C_true.shape)).astype(np.float32)
C_obs = torch.tensor(C_obs_np, device=device)

# Observations partielles: Cbb et Cccsf
OBS_MASK = torch.tensor([[1, 0, 1, 0]], device=device)

# =========================
# Losses
# =========================
def loss_data():
    C_pred = model(t_data)
    diff = (C_pred - C_obs) * OBS_MASK
    return (diff**2).mean()

def loss_ic():
    t_init = torch.tensor([[t0]], device=device, requires_grad=True)
    C0_pred = model(t_init)
    return (C0_pred**2).mean()

def sample_collocation(n=2000):
    tt = np.random.uniform(t0, t1, size=(n, 1)).astype(np.float32)
    return torch.tensor(tt, device=device)

def loss_ode(n_col=2000):
    t_col = sample_collocation(n_col).requires_grad_(True)
    C_pred = model(t_col)
    res = dCdt_autodiff(t_col, C_pred) - pbpk_rhs_torch(t_col, C_pred)
    return (res**2).mean()

# =========================
# Adam
# =========================
opt = torch.optim.Adam(list(model.parameters()) + [raw_PSbb, raw_PScsf], lr=1e-3)
w_data, w_ic, w_ode = 10.0, 1.0, 2.0

for it in range(1, 15001):
    opt.zero_grad()
    Ld = loss_data()
    Li = loss_ic()
    Lo = loss_ode(2000)
    loss = w_data*Ld + w_ic*Li + w_ode*Lo
    loss.backward()
    opt.step()

    if it % 3000 == 0:
        print(f"it={it:5d} | loss={loss.item():.4e} | Ld={Ld.item():.2e} Li={Li.item():.2e} Lo={Lo.item():.2e}")
        print_params("  snapshot")

print_params("After Adam (Qbb fixed)")

# =========================
# Plots
# =========================
with torch.no_grad():
    C_pred_np = model(t_data).cpu().numpy()

labels = ["Cbb", "Cbm", "Cccsf", "Cscsf"]

plt.figure()
for j in range(4):
    plt.plot(t_np, C_true[:, j], label=f"{labels[j]} true")
plt.legend(); plt.title("PBPK true (forward)"); plt.xlabel("t"); plt.show()

plt.figure()
for j in range(4):
    plt.plot(t_np, C_pred_np[:, j], label=f"{labels[j]} pred")
plt.legend(); plt.title("iPINN prediction (Qbb fixed)"); plt.xlabel("t"); plt.show()


## 5. Extension : ajout de quelques observations sur `Cbm`

Pour tester si un peu d'information supplémentaire suffit à améliorer l'identifiabilité, on ajoute 5 points d'observation sur le compartiment `Cbm` (initialement non observé), répartis au milieu de la fenêtre temporelle, en plus des observations continues sur `Cbb` et `Cccsf`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# Temps
# =========================
t0, t1 = 0.0, 48.0
N = 200
t_np = np.linspace(t0, t1, N).astype(np.float32)
t_data = torch.tensor(t_np.reshape(-1, 1), device=device)

# =========================
# Entrée plasma
# =========================
def Cart_np(t):
    return np.exp(-t / 8.0)

Cart_vals_np = Cart_np(t_np).astype(np.float32)
Cart_data = torch.tensor(Cart_vals_np.reshape(-1, 1), device=device)

def Cart_torch(t_query):
    tq = t_query.squeeze(1)
    tg = t_data.squeeze(1)
    idx = torch.searchsorted(tg, tq, right=False).clamp(1, len(tg) - 1)
    t0i = tg[idx - 1]; t1i = tg[idx]
    y0  = Cart_data.squeeze(1)[idx - 1]
    y1  = Cart_data.squeeze(1)[idx]
    w = (tq - t0i) / (t1i - t0i + 1e-12)
    y = y0 + w * (y1 - y0)
    return y.unsqueeze(1)

# =========================
# Paramètres FIXES
# =========================
Vbb, Vbm, Vccsf, Vscsf = 0.105, 1.25, 0.175, 0.055
Qcsf = 0.05

# =========================
# Paramètres à ESTIMER (bornés)
# =========================
def bounded(raw, lo, hi):
    return lo + (hi - lo) * torch.sigmoid(raw)

raw_PSbb  = nn.Parameter(torch.tensor(0.0, device=device))
raw_PScsf = nn.Parameter(torch.tensor(0.0, device=device))
raw_Qbb   = nn.Parameter(torch.tensor(0.0, device=device))

def get_params():
    return {
        "PSbb":  bounded(raw_PSbb,  0.05, 0.4),
        "PScsf": bounded(raw_PScsf, 0.02, 0.2),
        "Qbb":   bounded(raw_Qbb,   0.1,  1.0),
    }

true_params = {"PSbb": 0.15, "PScsf": 0.08, "Qbb": 0.5}

def print_params(tag):
    p = get_params()
    out = {k: float(p[k].detach().cpu()) for k in p}
    errs = {k: out[k] - true_params[k] for k in out}
    print(tag)
    print("  params:", out)
    print("  errors:", errs)

# =========================
# Réseau PINN
# =========================
class MLP(nn.Module):
    def __init__(self, width=64, depth=4):
        super().__init__()
        layers = [nn.Linear(1, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 4)]
        self.net = nn.Sequential(*layers)
    def forward(self, t):
        return self.net(t)

model = MLP().to(device)

def dCdt_autodiff(t, C_pred):
    grads = []
    for j in range(4):
        g = torch.autograd.grad(C_pred[:, j].sum(), t, create_graph=True, retain_graph=True)[0]
        grads.append(g)
    return torch.cat(grads, dim=1)

# =========================
# RHS PBPK stable
# =========================
def pbpk_rhs_torch(t, C):
    p = get_params()
    Cbb = C[:, 0:1]
    Cbm = C[:, 1:2]
    Ccc = C[:, 2:3]
    Csc = C[:, 3:4]
    Cart = Cart_torch(t)

    F_plasma = p["Qbb"] * (Cart - Cbb)
    F_bb_bm  = p["PSbb"] * (Cbb - Cbm)
    F_bm_cc  = p["PScsf"] * (Cbm - Ccc)
    F_cc_sc  = Qcsf * (Ccc - Csc)
    F_sc_out = Qcsf * Csc

    dCbb = (F_plasma - F_bb_bm) / Vbb
    dCbm = (F_bb_bm - F_bm_cc) / Vbm
    dCcc = (F_bm_cc - F_cc_sc) / Vccsf
    dCsc = (F_cc_sc - F_sc_out) / Vscsf

    return torch.cat([dCbb, dCbm, dCcc, dCsc], dim=1)

# =========================
# Données synthétiques (forward)
# =========================
C_true = np.zeros((N, 4), dtype=np.float32)
for i in range(N - 1):
    dt = t_np[i+1] - t_np[i]
    Cbb, Cbm, Ccc, Csc = C_true[i]
    Cart_i = Cart_np(t_np[i])

    F_plasma = true_params["Qbb"] * (Cart_i - Cbb)
    F_bb_bm  = true_params["PSbb"] * (Cbb - Cbm)
    F_bm_cc  = true_params["PScsf"] * (Cbm - Ccc)
    F_cc_sc  = Qcsf * (Ccc - Csc)
    F_sc_out = Qcsf * Csc

    dCbb = (F_plasma - F_bb_bm) / Vbb
    dCbm = (F_bb_bm - F_bm_cc) / Vbm
    dCcc = (F_bm_cc - F_cc_sc) / Vccsf
    dCsc = (F_cc_sc - F_sc_out) / Vscsf

    C_true[i+1] = C_true[i] + dt * np.array([dCbb, dCbm, dCcc, dCsc], dtype=np.float32)

sigma = 0.00
rng = np.random.default_rng(0)
C_obs_np = (C_true + sigma * rng.standard_normal(C_true.shape)).astype(np.float32)
C_obs = torch.tensor(C_obs_np, device=device)

# =========================
# OBSERVATIONS : Cbb + Ccc partout, + 5 points Cbm
# =========================
OBS_MASK = torch.zeros((N, 4), device=device)
OBS_MASK[:, 0] = 1.0  # Cbb
OBS_MASK[:, 2] = 1.0  # Cccsf

# 5 points de Cbm (choisis au milieu, hors t=0)
idx_bm = np.array([20, 50, 80, 110, 150], dtype=int)
OBS_MASK[idx_bm, 1] = 1.0

# =========================
# Losses
# =========================
def loss_data():
    C_pred = model(t_data)
    diff = (C_pred - C_obs) * OBS_MASK
    return (diff**2).mean()

def loss_ic():
    t_init = torch.tensor([[t0]], device=device, requires_grad=True)
    C0_pred = model(t_init)
    return (C0_pred**2).mean()

def sample_collocation(n=2000):
    tt = np.random.uniform(t0, t1, size=(n, 1)).astype(np.float32)
    return torch.tensor(tt, device=device)

def loss_ode(n_col=2000):
    t_col = sample_collocation(n_col).requires_grad_(True)
    C_pred = model(t_col)
    res = dCdt_autodiff(t_col, C_pred) - pbpk_rhs_torch(t_col, C_pred)
    return (res**2).mean()

# =========================
# Entraînement Adam
# =========================
opt = torch.optim.Adam(list(model.parameters()) + [raw_PSbb, raw_PScsf, raw_Qbb], lr=1e-3)
w_data, w_ic, w_ode = 10.0, 1.0, 2.0

for it in range(1, 15001):
    opt.zero_grad()
    Ld = loss_data()
    Li = loss_ic()
    Lo = loss_ode(2000)
    loss = w_data*Ld + w_ic*Li + w_ode*Lo
    loss.backward()
    opt.step()

    if it % 3000 == 0:
        print(f"it={it:5d} | loss={loss.item():.4e} | Ld={Ld.item():.2e} Li={Li.item():.2e} Lo={Lo.item():.2e}")
        print_params("  snapshot")

print_params("After Adam (+5 Cbm points)")

# =========================
# Plots
# =========================
with torch.no_grad():
    C_pred_np = model(t_data).cpu().numpy()

labels = ["Cbb", "Cbm", "Cccsf", "Cscsf"]

plt.figure()
for j in range(4):
    plt.plot(t_np, C_true[:, j], label=f"{labels[j]} true")
plt.legend(); plt.title("PBPK true (forward)"); plt.xlabel("t"); plt.show()

plt.figure()
for j in range(4):
    plt.plot(t_np, C_pred_np[:, j], label=f"{labels[j]} pred")
plt.scatter(t_np[idx_bm], C_true[idx_bm, 1], marker="x", s=60, label="Cbm measured (5 pts)")
plt.legend(); plt.title("iPINN prediction (+5 Cbm points)"); plt.xlabel("t"); plt.show()


## 6. Modèle PBPK final : estimation de 4 paramètres (avec amplitude `A`)

Dans le rapport, le modèle PBPK final comporte 4 paramètres libres : `πbb` (`PSbb`), `πcsf` (`PScsf`), `Qbb`, et l'amplitude `A` de l'injection intraveineuse `Cp(t) = A·e^(−αt)`. C'est cette version, la plus complète, qui produit les résultats finaux du rapport (Table 3, Figure 4) : erreur paramétrique moyenne de 12.0%, avec une bonne reconstruction globale malgré une légère sous-estimation du pic de concentration.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# =========================
# 0) Reproductibilité + device
# =========================
torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# 1) Temps
# =========================
t0, t1 = 0.0, 48.0
N = 200
t_np = np.linspace(t0, t1, N).astype(np.float32)
t_data = torch.tensor(t_np.reshape(-1, 1), device=device)

# =========================
# 2) Entrée plasma : base exp(-t/tau) + amplitude A
# =========================
tau = 8.0
A_true = 1.0  # amplitude vraie utilisée pour générer les données

def base_cart_np(t):
    return np.exp(-t / tau)

base_vals_np = base_cart_np(t_np).astype(np.float32)
base_data = torch.tensor(base_vals_np.reshape(-1, 1), device=device)

def base_cart_torch(t_query):
    tq = t_query.squeeze(1)
    tg = t_data.squeeze(1)
    idx = torch.searchsorted(tg, tq, right=False).clamp(1, len(tg) - 1)
    t0i = tg[idx - 1]; t1i = tg[idx]
    y0  = base_data.squeeze(1)[idx - 1]
    y1  = base_data.squeeze(1)[idx]
    w = (tq - t0i) / (t1i - t0i + 1e-12)
    y = y0 + w * (y1 - y0)
    return y.unsqueeze(1)

# =========================
# 3) Paramètres FIXES
# =========================
Vbb, Vbm, Vccsf, Vscsf = 0.105, 1.25, 0.175, 0.055
Qcsf = 0.05

# =========================
# 4) Paramètres à ESTIMER (bornés) + A
# =========================
def bounded(raw, lo, hi):
    return lo + (hi - lo) * torch.sigmoid(raw)

raw_PSbb  = nn.Parameter(torch.tensor(0.0, device=device))
raw_PScsf = nn.Parameter(torch.tensor(0.0, device=device))
raw_Qbb   = nn.Parameter(torch.tensor(0.0, device=device))
raw_A     = nn.Parameter(torch.tensor(0.0, device=device))  # NOUVEAU

def get_params():
    return {
        "PSbb":  bounded(raw_PSbb,  0.05, 0.4),
        "PScsf": bounded(raw_PScsf, 0.02, 0.2),
        "Qbb":   bounded(raw_Qbb,   0.1,  1.0),
        "A":     bounded(raw_A,     0.2,  2.0),  # NOUVEAU
    }

true_params = {"PSbb": 0.15, "PScsf": 0.08, "Qbb": 0.5, "A": A_true}

def print_params(tag):
    p = get_params()
    out = {k: float(p[k].detach().cpu()) for k in p}
    errs = {k: out[k] - true_params[k] for k in out}
    print(tag)
    print("  params:", out)
    print("  errors:", errs)

# =========================
# 5) Réseau PINN
# =========================
class MLP(nn.Module):
    def __init__(self, width=64, depth=4):
        super().__init__()
        layers = [nn.Linear(1, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 4)]
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        return self.net(t)

model = MLP().to(device)

def dCdt_autodiff(t, C_pred):
    grads = []
    for j in range(4):
        g = torch.autograd.grad(C_pred[:, j].sum(), t, create_graph=True, retain_graph=True)[0]
        grads.append(g)
    return torch.cat(grads, dim=1)

# =========================
# 6) RHS PBPK (avec A estimé)
# =========================
def pbpk_rhs_torch(t, C):
    p = get_params()
    Cbb = C[:, 0:1]
    Cbm = C[:, 1:2]
    Ccc = C[:, 2:3]
    Csc = C[:, 3:4]

    Cart = p["A"] * base_cart_torch(t)  # NOUVEAU : amplitude

    F_plasma = p["Qbb"] * (Cart - Cbb)
    F_bb_bm  = p["PSbb"] * (Cbb - Cbm)
    F_bm_cc  = p["PScsf"] * (Cbm - Ccc)
    F_cc_sc  = Qcsf * (Ccc - Csc)
    F_sc_out = Qcsf * Csc

    dCbb = (F_plasma - F_bb_bm) / Vbb
    dCbm = (F_bb_bm - F_bm_cc) / Vbm
    dCcc = (F_bm_cc - F_cc_sc) / Vccsf
    dCsc = (F_cc_sc - F_sc_out) / Vscsf

    return torch.cat([dCbb, dCbm, dCcc, dCsc], dim=1)

# =========================
# 7) Données synthétiques forward (avec A_true)
# =========================
C_true = np.zeros((N, 4), dtype=np.float32)
for i in range(N - 1):
    dt = t_np[i+1] - t_np[i]
    Cbb, Cbm, Ccc, Csc = C_true[i]
    Cart_i = A_true * base_cart_np(t_np[i])

    F_plasma = true_params["Qbb"] * (Cart_i - Cbb)
    F_bb_bm  = true_params["PSbb"] * (Cbb - Cbm)
    F_bm_cc  = true_params["PScsf"] * (Cbm - Ccc)
    F_cc_sc  = Qcsf * (Ccc - Csc)
    F_sc_out = Qcsf * Csc

    dCbb = (F_plasma - F_bb_bm) / Vbb
    dCbm = (F_bb_bm - F_bm_cc) / Vbm
    dCcc = (F_bm_cc - F_cc_sc) / Vccsf
    dCsc = (F_cc_sc - F_sc_out) / Vscsf

    C_true[i+1] = C_true[i] + dt*np.array([dCbb, dCbm, dCcc, dCsc], dtype=np.float32)

sigma = 0.00
rng = np.random.default_rng(0)
C_obs_np = (C_true + sigma * rng.standard_normal(C_true.shape)).astype(np.float32)
C_obs = torch.tensor(C_obs_np, device=device)

# Observations partielles : Cbb + Ccc
OBS_MASK = torch.zeros((N, 4), device=device)
OBS_MASK[:, 0] = 1.0
OBS_MASK[:, 2] = 1.0

# =========================
# 8) Losses
# =========================
def loss_data():
    C_pred = model(t_data)
    diff = (C_pred - C_obs) * OBS_MASK
    return (diff**2).mean()

def loss_ic():
    t_init = torch.tensor([[t0]], device=device, requires_grad=True)
    return (model(t_init)**2).mean()

def sample_collocation(n=2000):
    tt = np.random.uniform(t0, t1, size=(n, 1)).astype(np.float32)
    return torch.tensor(tt, device=device)

def loss_ode(n_col=2000):
    t_col = sample_collocation(n_col).requires_grad_(True)
    C_pred = model(t_col)
    res = dCdt_autodiff(t_col, C_pred) - pbpk_rhs_torch(t_col, C_pred)
    return (res**2).mean()

# =========================
# 9) Entraînement Adam
# =========================
opt = torch.optim.Adam(list(model.parameters()) + [raw_PSbb, raw_PScsf, raw_Qbb, raw_A], lr=1e-3)
w_data, w_ic, w_ode = 10.0, 1.0, 2.0

for it in range(1, 15001):
    opt.zero_grad()
    Ld = loss_data()
    Li = loss_ic()
    Lo = loss_ode(2000)
    loss = w_data*Ld + w_ic*Li + w_ode*Lo
    loss.backward()
    opt.step()

    if it % 3000 == 0:
        print(f"it={it:5d} | loss={loss.item():.4e} | Ld={Ld.item():.2e} Li={Li.item():.2e} Lo={Lo.item():.2e}")
        print_params("  snapshot")

print_params("After Adam (estimating A)")

# =========================
# 10) Plots
# =========================
with torch.no_grad():
    C_pred_np = model(t_data).cpu().numpy()

plt.figure()
for j, lab in enumerate(["Cbb","Cbm","Cccsf","Cscsf"]):
    plt.plot(t_np, C_true[:, j], label=f"{lab} true")
plt.legend(); plt.title("PBPK true"); plt.xlabel("t"); plt.show()

plt.figure()
for j, lab in enumerate(["Cbb","Cbm","Cccsf","Cscsf"]):
    plt.plot(t_np, C_pred_np[:, j], label=f"{lab} pred")
plt.legend(); plt.title("iPINN prediction (estimating A)"); plt.xlabel("t"); plt.show()


## Conclusion

Ce travail démontre l'efficacité des PINNs pour la modélisation PBPK :
- **Modèle linéaire** : reconstruction fidèle des profils de concentration, erreur paramétrique moyenne de **5.4%**.
- **Modèle PBPK** : reconstruction correcte avec une sous-estimation des transitoires rapides, erreur paramétrique moyenne de **12.0%**.
- **Identifiabilité** : une forte corrélation (ρ = 0.87) entre `πbb` et `Qbb` révèle une non-identifiabilité intrinsèque au modèle PBPK, bien que leur combinaison fonctionnelle (clairance totale) reste bien identifiée.

Ces résultats ouvrent des perspectives pour la médecine personnalisée et l'optimisation posologique, tout en soulignant l'importance de l'analyse d'identifiabilité dans les modèles compartimentaux complexes.